In [17]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [18]:
## create the embedding model  (embedding converts chunks to vectors)
embedding_function=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [19]:
## vectore store(store vectors presistently)
vectore_store=Chroma(
    embedding_function= HuggingFaceEmbeddings(
     model_name="sentence-transformers/all-MiniLM-L6-v2"
),
persist_directory="my_chroma_db",
collection_name="simple"
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [20]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma

pdf_path=r"C:\Users\Win 11\Desktop\Gen ai\chat_with_pdf project\uploads\Saloni_Agarwal_Resume.pdf"

loader=PyPDFLoader(pdf_path)
documents=loader.load()

In [21]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
recursive_char_splitter= RecursiveCharacterTextSplitter(chunk_size=50,chunk_overlap=10)
split_documents= recursive_char_splitter.split_documents(documents)

In [22]:
## adding our split documents
vectore_store.add_documents(documents=split_documents)


['9864076f-84ef-46bb-b029-48a7422bf20c',
 'fb3cc3f2-4d75-4fe5-bc6c-dedb1fbbdc77',
 'e346897d-def1-463d-908a-4e3923cf521f',
 '6c32cbc5-b699-43df-a991-4a7b01189e3c',
 '99f42844-f99c-488b-9d1b-9fb9ca11dce9',
 'a55e3f7b-4f56-4cf0-8bbb-fac4a5966795',
 'd0703890-901a-468f-aa11-9a85fbb2d0fd',
 '05e77fdb-03ca-4467-a685-b3673a0795e9',
 '66afb163-a46b-4625-8aaa-70b1b287bb4e',
 '9797f66d-ab54-453c-9162-b5e89c74670b',
 'b58e4b9b-a684-4c09-a6e0-57b2d7e28583',
 '88c86b09-f9c6-4b0a-aced-d18a6f073962',
 '50aa415d-a9cb-4d13-9835-6f08575ec2bd',
 '93a8ca13-3557-4ae5-afcf-7e15be1ee83a',
 '6b10ef4d-1d46-46d0-923c-04c1855e2bf3',
 '6d7a343b-d8fa-43c9-b22c-c4ce43bef9af',
 '02310eff-8053-40ed-9f1e-222447136dd5',
 '191506cf-b77c-433b-9664-efcbc924acdf',
 '2dd0fa7b-ecd7-4cc1-b951-a35af4562174',
 '4d4f292c-a520-4a85-9e58-b850397fe635',
 '44f31fc1-e7cd-4503-b6b9-70fea6153228',
 '3fb631bd-3ec3-4601-9468-2e613f51845d',
 '473222f6-e9b1-46a3-bbe9-fde5860802f9',
 '0b6607eb-11c3-49d6-ac42-dc630cfdfb77',
 '02cc1c31-d372-

In [23]:
## retriever(find the most releated chunks from th stored pdf)
retriever=vectore_store.as_retriever(search_kwargs={"k":3})  ## creates a search interface over your vectore store
question="what certificates is mentioned"
relevent_docs=retriever.invoke(question)  ## search the database using the question
for doc in relevent_docs:
    print(doc.page_content)



TRAINING & CERTIFICATIONS
TRAINING & CERTIFICATIONS
TRAINING & CERTIFICATIONS


In [24]:
## now creting prompt- retriver finds text but doesnt know how chatbot should answer so prompt tells the model to use the retrieved pdf text,answer and say no if dont know he ans

from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import PromptTemplate

test_prompt = PromptTemplate(
    template="""
You are a helpful assistant that answers questions using PDF content.

Rules:
1. Answer the question using ONLY the provided context.
2. Do not ask the user to provide information that is already in the context.
3. If the answer is not present in the context, say so.
4. Return only a valid JSON object matching this structure:
   - answer: string
   - found_in_document: boolean
   - source_pages: list of integers

Context:{context}

Question:{question}

Example JSON:{{"answer": "Python and C++", "found_in_document": true, "source_pages": [1]}}
""",
    input_variables=["context", "question"])

In [25]:
## ollama LLM- the retriver finds text but the text does not in a conversational answer then llm reads the question and retrived context and generate a response 

from langchain_ollama import ChatOllama
llm=ChatOllama(model="llama3.2:1b ",temperature=0)


In [26]:
## pydantic output parser- llm normally returns text but sometimes want a predictable structure so our output can display the answer and pdf pages no  seperately
from pydantic import BaseModel,Field
from langchain_core.output_parsers import PydanticOutputParser

class PDFAnswer(BaseModel):
    answer:str=Field(description="answer based on the pdf")
    found_in_document:bool=Field(description="whether yhe answer is supported by the document" )
    source_pages:list[int]=Field(description="relevant page number start from 1")

parser=PydanticOutputParser(pydantic_object=PDFAnswer)


prompt=PromptTemplate(template=""" you are a helpful pdf assistant. 
                                    answer the questions uing only the provided context . if answer was not supported by the context say set found_in_document to false,
                                    and return no source page
                                    context:{context}
                                    question:{question}
                                    {format_instructions}""",
                                    input_variables=["context","question"],
                                    partial_variables={"format_instructions":parser.get_format_instructions()}
)
                            



In [27]:
## connecting everything 
chain=test_prompt| llm|parser

In [12]:
## testing chain
result=chain.invoke({
    "context":""" python and c++ are programming lanuages """,
    "question":"which programing languages are mentioned?"
})

print(result.answer)
print(result.found_in_document)
print(result.source_pages)


Python and C++
True
[1]


In [13]:
retriever=vectore_store.as_retriever(search_kwargs={"k":3})

In [14]:
question = "What skills are mentioned in my resume?"

docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content for doc in docs
)

print("Retrieved PDF content:")
print(context)

Retrieved PDF content:
to present education, skills, projects, and

to present education, skills, projects, and

TRAINING & CERTIFICATIONS


In [15]:
## rag chain
from langchain_core.prompts import PromptTemplate

def answer_pdf_question(question):
    # 1. Retrieve relevant chunks from the PDF
    docs = retriever.invoke(question)

    # 2. Combine the retrieved text
    context = "\n\n".join(doc.page_content for doc in docs)

    # 3. Create the prompt
    formatted_prompt = test_prompt.invoke({
        "context": context,
        "question": question})

    # 4. Ask Ollama for an answer
    response = llm.invoke(formatted_prompt)

    # 5. Parse the response
    result = parser.invoke(response.content)

    return result


In [16]:
result = answer_pdf_question("What skills are mentioned in my resume?")

print("Answer:", result.answer)
print("Found in document:", result.found_in_document)
print("Source pages:", result.source_pages)

Answer: None
Found in document: False
Source pages: []
